# Responses API test (Scenario 02: non-streaming + streaming)

**QE Perspective:** We validate that the Responses API returns completed status and expected content for a simple prompt, and that **streaming** yields chunks that concatenate to the full message. This ensures the API contract (status, output shape, chunking) is stable and testable.

- **Non-streaming:** One-shot response; assert status and that the answer contains the expected fact (e.g. Paris).
- **Streaming:** Iterate the stream, capture full text; assert at least one chunk and that the concatenated message matches expectations.

Config: `base_url`, `model` from notebook_helpers or env (`BASE_URL`, `MODEL`). Run via pytest.


In [ ]:
# Setup
import os
from scripts.helpers import response_text

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("MODEL")
assert model, (
    "MODEL env var is required. Set it before running: export MODEL=your-model-id"
)

In [ ]:
from ogx_client import OgxClient

client = OgxClient(base_url=base_url)

In [ ]:
response = client.responses.create(
    model=model, input="What is the capital of France? Answer in one short sentence."
)
assert response.status == "completed", (
    f"Expected status completed, got {response.status}"
)
assert response.output, "Expected at least one output item"
text = response_text(response)
assert "Paris" in text, f"Expected Paris in response, got: {text}"

## Streaming: iterate stream and capture full message

QE: Validate chunking logic — stream yields chunks; concatenating them produces the full response.


In [ ]:
stream = client.responses.create(
    model=model,
    input="What is the capital of France? Answer in one short sentence.",
    stream=True,
)
chunks = []
full_text = ""
for chunk in stream:
    chunks.append(chunk)
    # Text deltas arrive as OutputTextDelta chunks with a .delta attribute
    delta = getattr(chunk, "delta", None)
    if isinstance(delta, str):
        full_text += delta
assert len(chunks) >= 1, "Expected at least one streamed chunk"
assert full_text.strip(), "Expected non-empty full message from stream"
assert "Paris" in full_text, (
    f"Expected Paris in streamed response, got: {full_text[:200]}"
)